In [ ]:
import os
import re
import json
import csv
from openai import OpenAI
from collections import Counter
from tqdm import tqdm
import pandas as pd
import time, random

api_key = ""
model_name = "gpt-5"
json_path = ""

client = OpenAI(api_key=api_key)

json_dir = ""
out_dir = "filtered_BR"
os.makedirs(out_dir, exist_ok=True)


In [2]:
# 固定的分类 prompt（保持原文不变，且把 JSON 插入到 "Do not output anything else." 之前）
base_prompt = """Your response MUST be a single word: Yes, No, or Cannot. Do not provide any explanation or extra text.
Use the following strict rules for classification:
1. Is it a Non-Crash Functional (NCF) Bug?
An NCF bug is when the app remains interactive but behaves incorrectly (e.g., shows wrong data, performs the wrong action).
If it is NOT an NCF bug, classify as No.
Reasons for No:
Crashes/Freezes: The report mentions "crash", "force close", "freeze", "hang", "ANR", or "unresponsive".
Not a Bug: It's a feature request, question, or discussion.
Out of Scope Platform: The bug is on iOS, Windows, desktop, a server, or a web browser. It must be specific to the Android app.
2. If it IS an NCF bug, can it be reproduced under specific constraints?
The bug must meet ALL of the following criteria to be considered reproducible.
If it IS an NCF bug AND meets ALL criteria below, classify as Yes.
If it IS an NCF bug BUT fails to meet EVEN ONE criterion below, classify as Cannot.
Criteria for a Yes Classification (All must be met):
Criterion A: Simple Reproducibility
Actions are limited to: click, long click, text input, scroll, swipe, and standard system navigation (back, home).
Reasons for Cannot: Requires device rotation, complex gestures, or changing system settings (e.g., language, proxy, permissions).
Criterion B: Text-Only Verification
The bug's outcome can be confirmed by on-screen text or the presence/absence of UI elements.
Reasons for Cannot: Verification requires checking:
Visuals: Colors, fonts, layouts, UI element positioning, graphical glitches, icons.
Media: The content of images/videos (e.g., a blurry picture), or anything related to audio/video playback state.
Criterion C: Single Android Device Requirement
The entire process must be performed on one Android device.
Reasons for Cannot: Requires interaction with a PC, server, web browser, another phone, or any external hardware/software.
Criterion D: Login Handling Logic
The bug is not a standard login failure with a clear error message (e.g., "wrong password", "server error").
Reasons for Cannot: The issue is a predictable login error. A login bug is only Yes if a previously working login fails unexpectedly.
Criterion E: No Significant Waiting
The bug appears immediately or within a few seconds.
Reasons for Cannot: Requires waiting for a long process like a large file upload/download, a full data sync, or a specific timeout.
Criterion F: No Notification Bar Interaction
The reproduction steps do not involve the Android notification bar or shade.
Reasons for Cannot: Requires opening, reading, or interacting with notifications.
Summary of single-word outputs:
Yes: It is an NCF bug AND it is reproducible under all criteria (A-F).
Cannot: It is an NCF bug BUT it is NOT reproducible under all criteria (A-F).
No: It is NOT an NCF bug at all (crash, feature request, wrong platform).
Do not output anything else.\n
"""


In [3]:
# 允许的标签 + 重试封装（放在 for json_file in ... 之前）
ALLOWED_LABELS = {"yes", "no", "cannot"}

def classify_with_retry(payload: str, max_retries: int = 3, base_sleep: float = 1.2) -> str:
    """
    仅在将要返回 error 的情况下触发重试（异常 / 空返回 / 非法输出）。
    成功返回只可能是 'yes'|'no'|'cannot' 之一；最终失败返回 'error'。
    """
    for attempt in range(1, max_retries + 1):
        try:
            resp = client.chat.completions.create(
                model=model_name,
                temperature=1,  # 你原来的设置保留
                messages=[
                    {"role": "system", "content": "You are an expert software quality assurance engineer. Your task is to analyze a bug report for an Android application and classify it. You will be given a JSON object containing the bug report's title and body."},
                    {"role": "user", "content": payload},
                ],
            )
            content = getattr(getattr(resp.choices[0], "message", None), "content", None)
            verdict_norm = (content or "").strip().lower()

            # 合法就直接返回；否则触发“error 路径”的重试
            if verdict_norm in ALLOWED_LABELS:
                return verdict_norm

        except Exception:
            # 异常也走“error 路径”的重试
            pass

        # 只有当将要返回 error 时，才进行重试（指数回退 + 抖动）
        if attempt < max_retries:
            sleep_s = base_sleep * (2 ** (attempt - 1)) + random.uniform(0, 0.4)
            time.sleep(sleep_s)

    # 最终失败
    return "error"


In [4]:
# 遍历目录下所有一级 .json 文件
json_files = [f for f in os.listdir(json_dir) if f.endswith(".json") and os.path.isfile(os.path.join(json_dir, f))]

# all_verdicts = []

for json_file in json_files:
    if json_file == "selected_issue_0006.json" or json_file == "selected_issue_0007.json" or json_file == "selected_issue_0001.json":
        print(f"skip {json_file}\n")
        continue
    json_path = os.path.join(json_dir, json_file)

    with open(json_path, "r", encoding="utf-8") as f:
        loaded = json.load(f)
    elements = loaded if isinstance(loaded, list) else [loaded]

    yes_items, cannot_items, no_items = [], [], []
    verdict_list = []

    for element in tqdm(elements, desc=f"Processing {json_file}", unit="item"):
        element_str = json.dumps(element, ensure_ascii=False)
        prompt_cls = base_prompt + "\n" + element_str + "Now, analyze the following JSON and provide your one-word classification:"

        label = classify_with_retry(prompt_cls, max_retries=3)  # <-- 在“error”情况下最多重试 3 次
        verdict_list.append(label)  # 每条样本都 push，保证长度对齐

        if label == "yes":
            yes_items.append(element)
        elif label == "cannot":
            cannot_items.append(element)
        elif label == "no":
            no_items.append(element)
        else:
            # label == "error" 不进任何桶
            pass
        
    # 保存 YES 文件
    m = re.search(r'(\d{4,})', os.path.basename(json_file))
    suffix = m.group(1) if m else os.path.splitext(json_file)[0]
    out_path = os.path.join(out_dir, f"filtered_BR_{suffix}_{model_name}.json")
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(yes_items, f, ensure_ascii=False, indent=2)

    out_verdicts = os.path.join(out_dir, f"verdicts_{suffix}_{model_name}.json")
    with open(out_verdicts, "w", encoding="utf-8") as f:
        json.dump(verdict_list, f, ensure_ascii=False, indent=2)

    count = Counter(v.lower() for v in verdict_list)
    print(f"{json_file} -> Yes: {count.get('yes', 0)}, Cannot: {count.get('cannot', 0)}, No: {count.get('no', 0)}, Error: {count.get('error', 0)}")

    # === CSV 部分 ===
    out_csv = os.path.join(out_dir, f"results_compare_{suffix}.csv")

    if not os.path.exists(out_csv):
        # 创建新 CSV
        with open(out_csv, "w", encoding="utf-8", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["index", model_name])  # 表头
            for idx, label in enumerate(verdict_list, start=1):
                writer.writerow([idx, label.lower()])
        print(f"Saved new CSV to {out_csv}")
    else:
        # 已存在，追加列
        df = pd.read_csv(out_csv)
        if len(df) != len(verdict_list):
            print(f"Warning: verdict_list 长度 {len(verdict_list)} 和 CSV 行数 {len(df)} 不一致！")
        df[model_name] = [label.lower() for label in verdict_list]
        df.to_csv(out_csv, index=False, encoding="utf-8")
        print(f"Appended results as new column '{model_name}' to {out_csv}")


skip selected_issue_0006.json

skip selected_issue_0007.json

skip selected_issue_0001.json



Processing selected_issue_0002.json: 100%|██████████| 1000/1000 [2:58:58<00:00, 10.74s/item] 


selected_issue_0002.json -> Yes: 368, Cannot: 389, No: 243, Error: 0
Saved new CSV to filtered_BR_new/results_compare_0002.csv


Processing samples_100.json: 100%|██████████| 100/100 [17:56<00:00, 10.77s/item]


samples_100.json -> Yes: 42, Cannot: 41, No: 17, Error: 0
Saved new CSV to filtered_BR_new/results_compare_samples_100.csv


...

selected_issue_0003.json -> Yes: 403, Cannot: 407, No: 190, Error: 0
Saved new CSV to filtered_BR_new/results_compare_0003.csv


Processing selected_issue_0004.json: 100%|██████████| 1000/1000 [2:03:59<00:00,  7.44s/item] 


selected_issue_0004.json -> Yes: 298, Cannot: 255, No: 447, Error: 0
Saved new CSV to filtered_BR_new/results_compare_0004.csv


Processing selected_issue_0005.json: 100%|██████████| 1000/1000 [3:10:57<00:00, 11.46s/item] 

selected_issue_0005.json -> Yes: 431, Cannot: 453, No: 116, Error: 0
Saved new CSV to filtered_BR_new/results_compare_0005.csv
